子图：把一部分节点和边封装成一个独立的graph，然后把这个graph当做另一个graph的节点使用。

可以理解为模块化设计，一个复杂的业务被拆成多个子流程，每个子流程负责自己的逻辑，最后由主流程进行编排

子图的主要价值：
1.模块化
2.复用
3.独立测试
4.独立状态设计
5.支持嵌套


子图的两种集成方式：主要区别是State的传递的不同
方式1：编译后的子图直接添加为节点，父图和子图存在相同的state，可以共享state数据
方式2：父图的节点函数中调用子图，父图和子图的state完全不同





In [1]:


"""


"""
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


# =========================
# 1. 定义主图 State
# =========================

class ParentState(TypedDict):
    applicant: str
    amount: float
    department: str
    analysis: str
    remaining_budget: float
    review: str


# =========================
# 2. 定义子图 State
# =========================

class PrecheckState(TypedDict):
    # 与主图共享的字段
    applicant: str
    amount: float
    department: str
    review: str

    # 子图内部字段
    analysis: str
    remaining_budget: float


# =========================
# 3. 子图节点 A：分析申请
# =========================

def analyze_expense(state: PrecheckState):
    print("\n[子图节点 A] 分析报销申请")

    analysis = (
        f"{state['department']}的{state['applicant']}申请报销"
        f"{state['amount']}元。"
    )

    return {
        "analysis": analysis
    }


# =========================
# 4. 子图节点 B：检查预算
# =========================

def check_budget(state: PrecheckState):
    print("[子图节点 B] 检查部门预算")

    # 模拟预算数据
    budgets = {
        "技术部": 10000.0,
        "市场部": 8000.0,
        "人事部": 5000.0,
    }

    remaining = budgets.get(state["department"], 0.0)

    if state["amount"] <= remaining:
        review = "预算充足，建议进入人工审批。"
    else:
        review = "预算不足，建议暂缓审批。"

    return {
        "remaining_budget": remaining,
        "review": review,
    }


# =========================
# 5. 构建并编译子图
# =========================

subgraph_builder = StateGraph(PrecheckState)

subgraph_builder.add_node("analyze", analyze_expense)
subgraph_builder.add_node("check_budget", check_budget)

subgraph_builder.add_edge(START, "analyze")
subgraph_builder.add_edge("analyze", "check_budget")
subgraph_builder.add_edge("check_budget", END)

precheck_graph = subgraph_builder.compile()


# =========================
# 6. 定义主图节点
# =========================

def receive_application(state: ParentState):
    print("[主图节点] 接收报销申请")

    return {}


def output_result(state: ParentState):
    print("[主图节点] 输出预审结果")

    print("申请分析：", state["analysis"])
    print("剩余预算：", state["remaining_budget"])
    print("审核建议：", state["review"])

    return {}


# =========================
# 7. 构建主图
# =========================

parent_builder = StateGraph(ParentState)

parent_builder.add_node("receive", receive_application)

# 关键：将编译后的子图直接添加为主图节点
parent_builder.add_node("precheck", precheck_graph)

parent_builder.add_node("output", output_result)

parent_builder.add_edge(START, "receive")
parent_builder.add_edge("receive", "precheck")
parent_builder.add_edge("precheck", "output")
parent_builder.add_edge("output", END)

parent_graph = parent_builder.compile()


# =========================
# 8. 执行主图
# =========================

if __name__ == "__main__":
    result = parent_graph.invoke({
        "applicant": "张三",
        "amount": 1200.0,
        "department": "技术部",
        "analysis": "",
        "remaining_budget": 0.0,
        "review": "",
    })

    print("\n最终 State：")
    print(result)


[主图节点] 接收报销申请

[子图节点 A] 分析报销申请
[子图节点 B] 检查部门预算
[主图节点] 输出预审结果
申请分析： 技术部的张三申请报销1200.0元。
剩余预算： 10000.0
审核建议： 预算充足，建议进入人工审批。

最终 State：
{'applicant': '张三', 'amount': 1200.0, 'department': '技术部', 'analysis': '技术部的张三申请报销1200.0元。', 'remaining_budget': 10000.0, 'review': '预算充足，建议进入人工审批。'}
